<a href="https://colab.research.google.com/github/Ane-Graciano/ihc/blob/main/Vers%C3%A3o_Compacta_IHC_filmes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Preparação do Ambiente e Modelos (Consolidado)

In [1]:
# Instalação e configuração do Ollama e modelos
!sudo apt-get update
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!uv pip install ollama rich pandas

import subprocess
import time
import requests

# Iniciar o servidor Ollama
subprocess.Popen("ollama serve", shell=True)

# Esperar o Ollama iniciar
for i in range(10):
    try:
        requests.get("http://localhost:11434")
        print("Ollama rodando")
        break
    except:
        time.sleep(2)

# Baixar o modelo de linguagem (qwen2.5:3b)
!for i in {1..5}; do ollama pull qwen2.5:3b && break || sleep 5; done


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,303 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InReleas

### Conexão do Banco de Dados via Google Drive

Este trecho de código é responsável por conectar o ambiente do Colab ao banco de dados SQLite armazenado no Google Drive.

Primeiro, o Google Drive é montado no ambiente utilizando drive.mount, permitindo acesso direto aos arquivos salvos na nuvem. Em seguida, é definido o caminho do banco de dados (DB_PATH), que aponta para o arquivo movies.db dentro da pasta do Drive.

In [2]:
# 1. baixar
# "https://drive.google.com/file/d/1lZzqoutfq4QZijBJRBzEAeCevipz_SP9/view?usp=sharing"
!gdown "https://drive.google.com/uc?id=1lZzqoutfq4QZijBJRBzEAeCevipz_SP9"

# 2. conectar
import sqlite3
conn = sqlite3.connect("moviesBackup.db")

# 3. usar normalmente
cursor = conn.cursor()
cursor.execute("SELECT * FROM movies LIMIT 5")
print(cursor.fetchall())

DATABASE_NAME = '/content/moviesBackup.db'

def get_db_connection():
    """Establishes a connection to the SQLite database and confirms success."""
    try:
        # Adicionado check_same_thread=False para compatibilidade com ThreadPoolExecutor
        conn = sqlite3.connect(DATABASE_NAME, check_same_thread=False)
        conn.row_factory = sqlite3.Row  # Permite acessar colunas pelo nome
        # print(f"Conexão com o banco '{DATABASE_NAME}' estabelecida com sucesso!") # Removido print para evitar logs excessivos
        return conn
    except sqlite3.Error as e:
        print(f"Erro ao conectar com o banco: {e}")
        return None

Downloading...
From (original): https://drive.google.com/uc?id=1lZzqoutfq4QZijBJRBzEAeCevipz_SP9
From (redirected): https://drive.google.com/uc?id=1lZzqoutfq4QZijBJRBzEAeCevipz_SP9&confirm=t&uuid=4ee21569-f39e-4207-968e-0db0843d9b39
To: /content/moviesBackup.db
100% 139M/139M [00:01<00:00, 101MB/s]
[(5492, 'Gunner', 'Gunner', '2024-08-16', 'Action|Thriller|Crime', 5.2, 210, 5.394, 'en', "While on a camping trip in order to reconnect, war veteran Colonel Lee Gunner must save his two sons from a gang of violent bikers when they're kidnapped after accidentally stumbling upon to a massive drug operation.", 'https://image.tmdb.org/t/p/w500/cS2TXN1YlrCvkZmMxaevC1ZKtEz.jpg', 'Dimitri Logothetis', 'N/A', None), (13494, 'Red Sonja', 'Red Sonja', '2025-07-31', 'Adventure|Action|Fantasy', 5.746, 183, 11.5262, 'en', 'A young girl rises from the ashes of tragedy to become the most feared warrior woman of all time: the She-Devil with a Sword.', 'https://image.tmdb.org/t/p/w500/aE3yh4y0h96CZZpLo0UDFM

### Criação das Ferramentas (Tools)
Nesta seção, declaramos as funções Python que farão consultas SQL no banco de dados. Elas serão transformadas nas 'ferramentas' da Inteligência Artificial.

In [3]:
import time
import sqlite3
import pandas as pd # Importar pandas para read_sql_query

# =========================
# TIMER (necessário para as tools)
# =========================
def agora():
    return time.perf_counter()

# =========================
# TOOLS (versões completas)
# =========================
def selectAll():
    t = agora()
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT title, release_date, genres, vote_average FROM movies LIMIT 10")
    rows = cursor.fetchall()
    conn.close()
    print("[DEBUG] Tempo selectAll:", round(agora() - t, 4), "s")
    return "\n".join(f"{r[0]} | {r[1]} | {r[2]} | {r[3]}" for r in rows)

def search_by_name(title: str):
    t = agora()
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT title, release_date, genres, vote_average FROM movies WHERE title LIKE ? LIMIT 10", (f"%{title}%",))
    rows = cursor.fetchall()
    conn.close()
    print("[DEBUG] Tempo search_by_name:", round(agora() - t, 4), "s")
    return "\n".join(f"TITULO: {r[0]} | DATA: {r[1]} | GENERO: {r[2]} | NOTA: {r[3]}" for r in rows)

def search_by_rating(vote_average: float) -> str:
    conn = get_db_connection()
    cursor = conn.cursor()

    query = "SELECT title, vote_average, genres FROM movies WHERE vote_average > ? ORDER BY vote_average DESC LIMIT 10"
    cursor.execute(query, (vote_average,))

    movies = cursor.fetchall()
    conn.close()

    if not movies:
        return "Nenhum filme encontrado com essa nota."

    return "\n".join(f"{row[0]} | Nota: {row[1]} | Gênero: {row[2]}" for row in movies)

def search_by_genre(genre: str):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT title, release_date, genres, vote_average FROM movies WHERE genres LIKE ? LIMIT 10", (f"%{genre}%",))
    rows = cursor.fetchall()
    conn.close()
    return "\n".join(f"{r[0]} | Gênero: {r[2]}" for r in rows)

def search_latest_releases(limit: int = 10) -> str:
    t = agora()
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT title, release_date, genres, vote_average
        FROM movies
        WHERE release_date IS NOT NULL
        ORDER BY release_date DESC
        LIMIT ?
    """, (limit,))

    rows = cursor.fetchall()
    conn.close()

    print("[DEBUG] Tempo search_latest_releases:", round(agora() - t, 4), "s")

    if not rows:
        return "Nenhum lançamento recente encontrado."

    return "\n".join(
        f"TITULO: {r[0]} | DATA: {r[1]} | GENERO: {r[2]} | NOTA: {r[3]}"
        for r in rows
    )

def search_by_year(year: str):
    t = agora()
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT title, release_date, genres, vote_average
        FROM movies
        WHERE release_date LIKE ?
        ORDER BY popularity DESC
        LIMIT 10
    """, (f"{year}%",))

    rows = cursor.fetchall()
    conn.close()

    print("[DEBUG] Tempo search_by_year:", round(agora() - t, 4), "s")

    if not rows:
        return f"Nenhum filme encontrado para o ano {year}."

    return "\n".join(
        f"TITULO: {r[0]} | DATA: {r[1]} | GENERO: {r[2]} | NOTA: {r[3]}"
        for r in rows
    )

def search_by_director(director_name: str) -> str:
    conn = get_db_connection()
    cursor = conn.cursor()

    query = """
        SELECT title, vote_average, genres
        FROM movies
        WHERE director LIKE ? AND director IS NOT 'N/A'
        ORDER BY vote_average DESC
        LIMIT 10
    """
    cursor.execute(query, (f"%{director_name}%",))

    movies = cursor.fetchall()
    conn.close()

    if not movies:
        return f"Nenhum filme encontrado para o diretor '{director_name}'."

    return "\n".join(f"{row[0]} | Nota: {row[1]} | Gênero: {row[2]}" for row in movies)

def search_by_streaming(provider_name: str) -> str:
    conn = get_db_connection()
    cursor = conn.cursor()

    query = """
        SELECT DISTINCT m.title, m.vote_average, m.genres
        FROM movies m
        JOIN streaming_providers s ON m.id = s.movie_id
        WHERE s.provider_name LIKE ?
        ORDER BY m.vote_average DESC
        LIMIT 10
    """

    cursor.execute(query, (f"%{provider_name}%",))

    movies = cursor.fetchall()
    conn.close()

    if not movies:
        return f"Nenhum filme encontrado disponível no streaming '{provider_name}'."

    return "\n".join(f"{row[0]} | Nota: {row[1]} | Gênero: {row[2]}" for row in movies)


### O Agente de IA e Interface de Chat
Orquestração do Agente: recebe a pergunta, decide qual ferramenta usar, coleta os dados do banco e gera a resposta final na interface gráfica.

In [4]:
import ipywidgets as widgets
from IPython.display import display, HTML
from ollama import chat
import json

# =========================
# DATABASE PATH
# =========================
# DB_PATH é definido em uma célula anterior, mas garantindo que esteja disponível aqui para o agente.
# Caso esteja em um notebook separado, seria necessário definir aqui: DB_PATH = "movies.db"

# =========================
# MODEL
# =========================
model = "qwen2.5:3b"

# =========================
# MEMORY
# =========================
messages = [
    {
        "role": "system",
        "content": (
            "Você é um assistente rápido de filmes.\n"
            "Regras:\n"
            "- Sempre use tools para dados de filmes\n"
            "- Nunca invente filmes\n"
            "- Responda de forma curta e direta\n"
        )
    }
]

# =========================
# TOOLS DEFINITION FOR AGENT
# =========================
tools = [
    {
        "type": "function",
        "function": {
            "name": "selectAll",
            "description": "Retorna filmes do banco",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_by_name",
            "description": "Busca filmes por título",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"}
                },
                "required": ["title"]
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "search_latest_releases",
        "description": "Retorna os filmes mais recentes do banco, ordenados por data de lançamento.",
        "parameters": {
            "type": "object",
            "properties": {
                "limit": {
                    "type": "integer",
                    "description": "Quantidade de filmes a retornar (padrão: 10)"
                }
            }
        }
    }
},
    {
        "type": "function",
        "function": {
            "name": "search_by_genre",
            "description": "Busca filmes por gênero cinematográfico (ex: Action, Comedy, Drama)",
            "parameters": {
                "type": "object",
                "properties": {
                    "genre": {"type": "string"}
                },
                "required": ["genre"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_by_rating",
            "description": "Busca filmes com avaliação acima do valor informado (ex: 7.5, 8.0)",
            "parameters": {
                "type": "object",
                "properties": {
                    "vote_average": {"type": "number"}
                },
                "required": ["vote_average"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_by_year",
            "description": "Busca filmes pelo ano exato de lançamento.",
            "parameters": {
                "type": "object",
                "properties": {
                    "year": {
                        "type": "string",
                        "description": "O ano de lançamento do filme com 4 dígitos numéricos (exemplo: '2022', '2024')."
                    }
                },
                "required": ["year"]
            }
        }
    },
    {
      "type": "function",
      "function": {
          "name": "search_by_director",
          "description": "Busca filmes dirigidos por um diretor específico (ex: Christopher Nolan, Steven Spielberg)",
          "parameters": {
              "type": "object",
              "properties": {
                  "director_name": {"type": "string"}
              },
              "required": ["director_name"]
          }
      }
    },
    {
      "type": "function",
      "function": {
          "name": "search_by_streaming",
          "description": "Busca filmes disponíveis em uma plataforma de streaming específica (ex: Netflix, Amazon Prime Video, Disney Plus, Max)",
          "parameters": {
              "type": "object",
              "properties": {
                  "provider_name": {"type": "string"}
              },
              "required": ["provider_name"]
          }
      }
    }
]

# =========================
# UI COMPONENTS
# =========================
chat_history = widgets.Output(layout={
    "border": "1px solid #ddd",
    "height": "500px",
    "overflow_y": "auto",
    "padding": "10px",
    "margin_top": "10px"
})

input_box = widgets.Text(placeholder="Pergunte sobre filmes...", layout=widgets.Layout(width='70%'))
button = widgets.Button(description="Enviar", button_style="primary")

loader = widgets.HTML(
    value='''
    <div style="display: flex; align-items: center; margin-left: 10px;">
        <div class="spinner" style=
            "border: 3px solid #f3f3f3;
            border-top: 3px solid #3498db;
            border-radius: 50%;
            width: 18px;
            height: 18px;
            animation: spin 1s linear infinite;">
        </div>
        <style>@keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }</style>
    </div>
    ''',
    layout=widgets.Layout(visibility='hidden')
)

display(widgets.VBox([
    widgets.HBox([input_box, button, loader]),
    chat_history
]))

def render(role, text):
    label = "Você" if role == "user" else "Assistente"
    color = "#f0f0f0" if role == "user" else "#e3f2fd"

    html = f"""
    <div style="margin:8px; background-color:{color}; padding:12px; border-radius:12px; border: 1px solid #d1d1d1;">
        <b style="color: #555;">{label}:</b><br>
        <div style="white-space: pre-wrap; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin-top:5px;">{text}</div>
    </div>
    """
    with chat_history:
        display(HTML(html))

# =========================
# EXECUTOR
# =========================
def execute_tool(tool_call):
    nome = tool_call.function.name
    args = tool_call.function.arguments
    print("[DEBUG] Tool chamada:", nome)
    if isinstance(args, str):
        args = json.loads(args or "{}")
    func = globals().get(nome)
    return func(**args) if func else "tool não encontrada"

# =========================
# AGENT CORE
# =========================
def run_agent(user_input):
    t_start = agora()
    render("user", user_input)
    messages.append({"role": "user", "content": user_input})

    # Limita contexto para manter velocidade
    if len(messages) > 6:
        messages[:] = [messages[0]] + messages[-5:]

    # Chamada LLM 1 (Decisão)
    t1 = agora()
    response = chat(
        model=model,
        messages=messages,
        tools=tools,
        options={"temperature": 0, "num_predict": 180}
    )
    print("[DEBUG] Tempo LLM1:", round(agora() - t1, 4), "s")

    msg = response.message
    tool_calls = getattr(msg, "tool_calls", None)

    if not tool_calls:
        render("assistant", msg.content or "Não encontrei informações.")
        messages.append({"role": "assistant", "content": msg.content or ""})
        return

    # Processamento de Tools
    messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": tool_calls})

    for tool_call in tool_calls:
        resultado = execute_tool(tool_call)
        messages.append(
            {
                "role": "tool",
                "name": tool_call.function.name,
                "content": str(resultado)
            }
        )

    # Chamada LLM 2 (Resposta Final)
    t2 = agora()
    response2 = chat(
        model=model,
        messages=messages[-6:],
        options={"temperature": 0, "num_predict": 200}
    )
    print("[DEBUG] Tempo LLM2:", round(agora() - t2, 4), "s")

    final = response2.message.content
    render("assistant", final)
    messages.append({"role": "assistant", "content": final})
    print("[DEBUG] Tempo Total do Ciclo:", round(agora() - t_start, 4), "s")

# =========================
# EVENTS
# =========================
def on_click(_):
    texto = input_box.value.strip()
    if texto:
        input_box.value = ""
        # UI Feedback
        loader.layout.visibility = 'visible'
        button.disabled = True
        input_box.disabled = True

        try:
            run_agent(texto)
        finally:
            loader.layout.visibility = 'hidden'
            button.disabled = False
            input_box.disabled = False

button.on_click(on_click)
input_box.on_submit(on_click)

print("Sistema de busca de filmes pronto.")

Sistema de busca de filmes pronto.
[DEBUG] Tempo LLM1: 13.3533 s
[DEBUG] Tool chamada: search_by_genre
[DEBUG] Tempo LLM2: 77.1166 s
[DEBUG] Tempo Total do Ciclo: 90.4832 s


## TESTE AUTOMÁTICO DO AGENTE

### search_by_name

In [5]:
run_agent("Tem algum filme chamado Inception no banco?")

[DEBUG] Tempo LLM1: 56.3812 s
[DEBUG] Tool chamada: search_by_name
[DEBUG] Tempo search_by_name: 10.2571 s
[DEBUG] Tempo LLM2: 13.8952 s
[DEBUG] Tempo Total do Ciclo: 80.5426 s


In [ ]:
run_agent("Procura filmes com o nome Matrix")

[DEBUG] Tempo LLM1: 22.0309 s
[DEBUG] Tool chamada: search_by_name
[DEBUG] Tempo search_by_name: 0.0461 s
[DEBUG] Tempo LLM2: 143.5478 s
[DEBUG] Tempo Total do Ciclo: 165.6357 s


In [ ]:
run_agent("Existe algum filme com ‘Batman’ no título?")

[DEBUG] Tempo LLM1: 97.0507 s
[DEBUG] Tool chamada: search_by_name
[DEBUG] Tempo search_by_name: 0.0888 s
[DEBUG] Tempo LLM2: 221.3263 s
[DEBUG] Tempo Total do Ciclo: 318.4774 s


In [ ]:
run_agent("Quais filmes têm ‘Avengers’ no nome?")

[DEBUG] Tempo LLM1: 107.3675 s
[DEBUG] Tool chamada: search_by_name
[DEBUG] Tempo search_by_name: 0.0998 s
[DEBUG] Tempo LLM2: 237.6316 s
[DEBUG] Tempo Total do Ciclo: 345.1113 s


### search_by_rating

In [ ]:
run_agent("Me mostra os filmes mais bem avaliados acima de 7.5")

[DEBUG] Tempo LLM1: 104.9925 s
[DEBUG] Tool chamada: search_by_rating
[DEBUG] Tempo LLM2: 193.4173 s
[DEBUG] Tempo Total do Ciclo: 298.5354 s


In [ ]:
run_agent("Quais filmes são considerados os melhores do banco?")

[DEBUG] Tempo LLM1: 109.4917 s


In [ ]:
run_agent("Lista filmes com rating maior que 9")

[DEBUG] Tempo LLM1: 92.8587 s
[DEBUG] Tool chamada: search_by_rating
[DEBUG] Tempo LLM2: 146.7866 s
[DEBUG] Tempo Total do Ciclo: 239.7372 s


### search_by_genre

In [ ]:
run_agent("Quais filmes de ação você tem?")

[DEBUG] Tempo LLM1: 72.127 s
[DEBUG] Tool chamada: search_by_genre
[DEBUG] Tempo LLM2: 156.2396 s
[DEBUG] Tempo Total do Ciclo: 228.3792 s


In [ ]:
run_agent("Me mostra filmes de comédia")

[DEBUG] Tempo LLM1: 69.1445 s
[DEBUG] Tool chamada: search_by_genre
[DEBUG] Tempo LLM2: 151.8985 s
[DEBUG] Tempo Total do Ciclo: 221.059 s


In [ ]:
run_agent("Tem algum drama no banco?")

[DEBUG] Tempo LLM1: 66.7534 s
[DEBUG] Tool chamada: search_by_genre
[DEBUG] Tempo LLM2: 134.8761 s
[DEBUG] Tempo Total do Ciclo: 201.6453 s


In [ ]:
run_agent("Lista filmes de ficção científica")

[DEBUG] Tempo LLM1: 58.2849 s
[DEBUG] Tool chamada: search_by_genre
[DEBUG] Tempo LLM2: 119.0341 s
[DEBUG] Tempo Total do Ciclo: 177.4403 s


### search_latest_releases

In [ ]:
run_agent("Quais são os filmes mais recentes?")

[DEBUG] Tempo LLM1: 46.0838 s
[DEBUG] Tool chamada: search_latest_releases
[DEBUG] Tempo search_latest_releases: 0.1542 s
[DEBUG] Tempo LLM2: 127.6223 s
[DEBUG] Tempo Total do Ciclo: 173.8737 s


In [ ]:
run_agent("Me mostra os últimos lançamentos")

[DEBUG] Tempo LLM1: 118.2265 s


In [ ]:
run_agent("O que saiu mais novo no banco?")

[DEBUG] Tempo LLM1: 139.5468 s


In [ ]:
run_agent("Lista os filmes em ordem de lançamento")

[DEBUG] Tempo LLM1: 58.4691 s
[DEBUG] Tool chamada: search_latest_releases
[DEBUG] Tempo search_latest_releases: 0.1188 s
[DEBUG] Tempo LLM2: 142.7143 s
[DEBUG] Tempo Total do Ciclo: 201.3135 s


### search_by_year

In [ ]:
run_agent("Quais filmes de 2023 você tem?")

[DEBUG] Tempo LLM1: 70.8382 s
[DEBUG] Tool chamada: search_by_year
[DEBUG] Tempo search_by_year: 0.1296 s
[DEBUG] Tempo LLM2: 190.7061 s
[DEBUG] Tempo Total do Ciclo: 261.6871 s


In [ ]:
run_agent("Me mostra filmes lançados em 2022")

[DEBUG] Tempo LLM1: 106.3156 s
[DEBUG] Tool chamada: search_by_year
[DEBUG] Tempo search_by_year: 0.127 s
[DEBUG] Tempo LLM2: 227.7653 s
[DEBUG] Tempo Total do Ciclo: 334.2178 s


In [ ]:
run_agent("O que tem de 2024 no banco?")

[DEBUG] Tempo LLM1: 118.7564 s


### search_by_director

In [ ]:
run_agent("Quais filmes do Christopher Nolan?")

[DEBUG] Tempo LLM1: 110.9045 s
[DEBUG] Tool chamada: search_by_director
[DEBUG] Tempo LLM2: 82.1829 s
[DEBUG] Tempo Total do Ciclo: 193.1908 s


In [ ]:
run_agent("Tem filmes do Steven Spielberg?")

[DEBUG] Tempo LLM1: 34.6311 s
[DEBUG] Tool chamada: search_by_director
[DEBUG] Tempo LLM2: 73.2199 s
[DEBUG] Tempo Total do Ciclo: 108.0158 s


In [ ]:
run_agent("Me mostra filmes dirigidos por Quentin Tarantino")

[DEBUG] Tempo LLM1: 39.8488 s
[DEBUG] Tool chamada: search_by_director
[DEBUG] Tempo LLM2: 77.5199 s
[DEBUG] Tempo Total do Ciclo: 117.521 s


### search_by_streaming

In [ ]:
run_agent("Quais filmes estão na Netflix?")

[DEBUG] Tempo LLM1: 37.9388 s
[DEBUG] Tool chamada: search_by_streaming
[DEBUG] Tempo LLM2: 124.3806 s
[DEBUG] Tempo Total do Ciclo: 162.364 s


In [ ]:
run_agent("Tem algo disponível no Amazon Prime?")

[DEBUG] Tempo LLM1: 72.4901 s
[DEBUG] Tool chamada: search_by_streaming
[DEBUG] Tempo LLM2: 158.7857 s
[DEBUG] Tempo Total do Ciclo: 231.3004 s


In [ ]:
run_agent("O que posso assistir na Disney Plus?")

[DEBUG] Tempo LLM1: 71.4838 s
[DEBUG] Tool chamada: search_by_streaming
[DEBUG] Tempo LLM2: 164.2644 s
[DEBUG] Tempo Total do Ciclo: 235.7676 s
